## Gom nhóm ảnh

In [1]:
import paddle
print("1. Phiên bản Paddle:", paddle.__version__)
print("2. Có hỗ trợ CUDA không:", paddle.is_compiled_with_cuda())
print("3. Thiết bị đang sử dụng:", paddle.get_device())

1. Phiên bản Paddle: 3.0.0-beta1
2. Có hỗ trợ CUDA không: True
3. Thiết bị đang sử dụng: gpu:0


In [6]:
import os
import cv2
import shutil
import numpy as np
from PIL import Image
from paddleocr import PaddleOCR
import logging

# ================= CẤU HÌNH HỆ THỐNG & NGƯỠNG =================
os.environ['CUDA_VISIBLE_DEVICES'] = '-1' # Ép dùng CPU cho ổn định
logging.getLogger("ppocr").setLevel(logging.ERROR)

SKEW_THRESHOLD = 3.0       # Ưu tiên 1: Nghiêng > 3 độ
CURVE_VAR_THRESHOLD = 12.0 # Ưu tiên 2: Biến thiên góc (Ảnh cong) -> Hạ xuống 12 để nhạy hơn
BLUR_THRESHOLD = 40.0      # Ưu tiên 3: Độ mờ
MIN_BOX_COUNT = 30         # Ưu tiên 4: Số dòng chữ > 30 (Dày đặc)
SMALL_TEXT_RATIO = 0.025   # Ưu tiên 4: Chiều cao chữ < 2.5% ảnh (Chữ nhỏ)

# Khởi tạo PaddleOCR
ocr = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=False, show_log=False)

def preprocess_and_convert(source_folder):
    """Chuyển đổi toàn bộ WebP sang JPG trước khi xử lý"""
    webp_files = [f for f in os.listdir(source_folder) if f.lower().endswith('.webp')]
    if not webp_files: return
    
    print(f"--- Đang chuẩn hóa {len(webp_files)} ảnh WebP sang JPG ---")
    for f in webp_files:
        try:
            webp_path = os.path.join(source_folder, f)
            img = Image.open(webp_path).convert('RGB')
            jpg_path = os.path.join(source_folder, os.path.splitext(f)[0] + ".jpg")
            img.save(jpg_path, "JPEG", quality=95)
            os.remove(webp_path) # Xóa file WebP gốc để tránh xử lý trùng
        except: continue

def analyze_image(file_path):
    img = cv2.imread(file_path)
    if img is None: return None
    H, W = img.shape[:2]
    
    # 1. Lấy thông tin từ AI
    result = ocr.ocr(file_path, cls=True, rec=False)
    
    # 2. Các chỉ số vật lý
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()

    if not result or result[0] is None:
        return 'blur' if blur_score < BLUR_THRESHOLD else 'no_text'

    boxes = result[0]
    angles = []
    box_heights = []

    for box in boxes:
        if isinstance(box, list) and len(box) >= 4:
            try:
                # Toạ độ 4 góc: p1(trên-trái), p2(trên-phải), p3(dưới-phải), p4(dưới-trái)
                p1, p2, p4 = box[0], box[1], box[3]
                
                # Tính góc dòng chữ
                angle = np.degrees(np.arctan2(p2[1] - p1[1], p2[0] - p1[0]))
                angles.append(angle)
                
                # Tính chiều cao chữ (khoảng cách p1-p4)
                h_box = np.sqrt((p4[0] - p1[0])**2 + (p4[1] - p1[1])**2)
                box_heights.append(h_box)
            except: continue
    
    if not angles: return 'no_text'

    avg_angle = np.mean(angles)
    angle_std = np.std(angles) # Đây là chìa khóa để bắt ảnh cong
    avg_box_height_ratio = np.mean(box_heights) / H

    # ================= KIỂM TRA THEO THỨ TỰ ƯU TIÊN =================
    
    # 1. NGHIÊNG: Tất cả các dòng đều nghiêng cùng một hướng
    if abs(avg_angle) > SKEW_THRESHOLD:
        return 'skewed'
    
    # 2. CONG: Các dòng chữ có góc khác nhau (chữ uốn lượn trên chai lọ)
    if angle_std > CURVE_VAR_THRESHOLD:
        return 'curved'
    
    # 3. MỜ: 
    if blur_score < BLUR_THRESHOLD:
        return 'blur'
    
    # 4. CHỮ NHỎ / DÀY ĐẶC
    if len(boxes) > MIN_BOX_COUNT or avg_box_height_ratio < SMALL_TEXT_RATIO:
        return 'dense'
        
    # 5. CHUẨN
    return 'normal'

def main_process(folder_path):
    target_dirs = {
        'skewed': '1_Anh_Nghieng', 'curved': '2_Anh_Cong',
        'blur': '3_Anh_Mo', 'dense': '4_Chu_Nho_Day',
        'normal': '5_Anh_Chuan', 'no_text': '6_Khong_Co_Chu'
    }
    
    for d in target_dirs.values():
        os.makedirs(os.path.join(folder_path, d), exist_ok=True)

    preprocess_and_convert(folder_path)
    
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Đang phân loại {len(files)} ảnh...")

    for filename in files:
        file_path = os.path.join(folder_path, filename)
        category = analyze_image(file_path)
        if category:
            shutil.move(file_path, os.path.join(folder_path, target_dirs[category], filename))
            print(f"[{category.upper()}] {filename}")

if __name__ == "__main__":
    main_process("en")

Đang phân loại 588 ảnh...
[BLUR] 20210831_123238.jpg
[BLUR] 20210831_123245.jpg
[BLUR] 20210831_123251.jpg
[BLUR] 20210831_123309.jpg
[BLUR] 20210831_130450.jpg
[BLUR] 20210831_130523.jpg
[SKEWED] 20210831_130526.jpg
[BLUR] 20210831_130531.jpg
[SKEWED] 20210831_130538.jpg
[NORMAL] 20210831_130542.jpg
[SKEWED] 20210831_130548.jpg
[SKEWED] 20210831_130551.jpg
[SKEWED] 20210831_130555.jpg
[BLUR] 20210831_130722.jpg
[SKEWED] 20210831_130725.jpg
[SKEWED] 20210831_130737.jpg
[BLUR] 20210831_130740.jpg
[BLUR] 20210831_130743.jpg
[BLUR] 20210831_130746.jpg
[SKEWED] 20210831_130748.jpg
[SKEWED] 20210831_130749.jpg
[BLUR] 20210831_130752.jpg
[DENSE] 20210831_130753.jpg
[SKEWED] 20210831_130756.jpg
[SKEWED] 20210831_130758.jpg
[SKEWED] 20210831_130803.jpg
[NORMAL] 20210831_130805.jpg
[SKEWED] 20210831_130808.jpg
[NORMAL] 20210831_130812.jpg
[SKEWED] 20210831_130918.jpg
[SKEWED] 20210831_130934.jpg
[SKEWED] 20210831_130936.jpg
[BLUR] 20210831_132039.jpg
[SKEWED] 20210831_132044.jpg
[DENSE] 2021083

In [ ]:
import os
from PIL import Image
from pathlib import Path

def convert_webp_to_jpg(source_dir, target_dir, quality=95):
    """
    source_dir: Thư mục chứa ảnh .webp
    target_dir: Thư mục lưu ảnh .jpg sau khi chuyển
    quality: Chất lượng ảnh JPG (1-100), nên để cao để test OCR
    """
    # Tạo thư mục đích nếu chưa có
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    # Lấy danh sách tất cả file .webp
    files = [f for f in os.listdir(source_dir) if f.lower().endswith('.webp')]
    
    print(f"Tìm thấy {len(files)} file WebP. Đang bắt đầu chuyển đổi...")

    for filename in files:
        try:
            # Mở file WebP
            file_path = os.path.join(source_dir, filename)
            img = Image.open(file_path)

            # Chuyển đổi sang RGB (vì WebP có thể có hệ màu RGBA không lưu được vào JPG)
            img = img.convert('RGB')

            # Đổi đuôi file sang .jpg
            target_filename = Path(filename).stem + ".jpg"
            target_path = os.path.join(target_dir, target_filename)

            # Lưu file
            img.save(target_path, "JPEG", quality=quality)
            print(f"Thành công: {filename} -> {target_filename}")
            
        except Exception as e:
            print(f"Lỗi khi chuyển file {filename}: {e}")

    print("\n--- HOÀN THÀNH ---")
    print(f"Ảnh đã được lưu tại: {target_dir}")

if __name__ == "__main__":
   
    SOURCE = "vn" 
    TARGET = "images\\vn"
    
    convert_webp_to_jpg(SOURCE, TARGET)

Tìm thấy 250 file WebP. Đang bắt đầu chuyển đổi...
Thành công: 00000965_augmentin_250mg_5058_5bff_large_ad403504ac.webp -> 00000965_augmentin_250mg_5058_5bff_large_ad403504ac.jpg
Thành công: 00000967_augmentin_625mg_6579_63aa_large_ef4ff4d498.webp -> 00000967_augmentin_625mg_6579_63aa_large_ef4ff4d498.jpg
Thành công: 00001057_azimax_500mg_2149_5b71_large_d7fe898305.webp -> 00001057_azimax_500mg_2149_5b71_large_d7fe898305.jpg
Thành công: 00001403_bonlutin_6944_62bd_large_28df040743.webp -> 00001403_bonlutin_6944_62bd_large_28df040743.jpg
Thành công: 00001626_casodex_50mg_9841_5b6d_large_5bf8f3aebe.webp -> 00001626_casodex_50mg_9841_5b6d_large_5bf8f3aebe.jpg
Thành công: 00001640_cebraton_tang_cuong_tri_nho_6984_63ab_large_f8e788ed93.webp -> 00001640_cebraton_tang_cuong_tri_nho_6984_63ab_large_f8e788ed93.jpg
Thành công: 00001698_cefixim_500mg_goi_6319_5b69_large_ad8ca68591.webp -> 00001698_cefixim_500mg_goi_6319_5b69_large_ad8ca68591.jpg
Thành công: 00002069_contractubex_10g_7960_62a7_lar

In [ ]:
import os

def check_and_rename_files(parent_folder):
    """
    Quét tất cả thư mục con, tìm file có dấu cách và đổi tên.
    """
    count_found = 0
    count_renamed = 0

    print(f"--- Bắt đầu kiểm tra tại: {parent_folder} ---")

    # os.walk sẽ đi xuyên qua tất cả thư mục con
    for root, dirs, files in os.walk(parent_folder):
        for filename in files:
            # Kiểm tra xem có dấu cách trong tên file không
            if ' ' in filename:
                count_found += 1
                old_path = os.path.join(root, filename)
                
                # Tạo tên mới: thay dấu cách bằng gạch dưới
                new_filename = filename.replace(' ', '_')
                new_path = os.path.join(root, new_filename)
                
                # Kiểm tra nếu tên mới đã tồn tại thì đính thêm số để tránh ghi đè
                if os.path.exists(new_path):
                    name, ext = os.path.splitext(new_filename)
                    new_path = os.path.join(root, f"{name}_1{ext}")

                try:
                    os.rename(old_path, new_path)
                    print(f"[RENAME] '{filename}' -> '{os.path.basename(new_path)}'")
                    count_renamed += 1
                except Exception as e:
                    print(f"[ERROR] Không thể đổi tên {filename}: {e}")

    print("-" * 30)
    print(f"Hoàn thành!")
    print(f"Tổng số file có dấu cách tìm thấy: {count_found}")
    print(f"Đã xử lý đổi tên thành công: {count_renamed}")

if __name__ == "__main__":
    # Thay 'en_images' bằng thư mục gốc chứa 6 thư mục con của bạn
    BASE_DIR = "en" 
    check_and_rename_files(BASE_DIR)

--- Bắt đầu kiểm tra tại: en ---
------------------------------
Hoàn thành!
Tổng số file có dấu cách tìm thấy: 0
Đã xử lý đổi tên thành công: 0


: 

In [1]:
import os

def count_images_in_dataset(base_folders):
    # Các định dạng ảnh chúng ta đã chuẩn hóa
    VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.jfif')
    
    print(f"{'Thư mục gốc':<15} | {'Thư mục con':<20} | {'Số lượng ảnh'}")
    print("-" * 55)
    
    total_all = 0
    
    for base in base_folders:
        if not os.path.exists(base):
            print(f"![Cảnh báo] Không tìm thấy thư mục: {base}")
            continue
            
        base_total = 0
        # Lấy danh sách thư mục con và sắp xếp theo tên (1, 2, 3...)
        subdirs = sorted([d for d in os.listdir(base) if os.path.isdir(os.path.join(base, d))])
        
        for subdir in subdirs:
            subdir_path = os.path.join(base, subdir)
            # Đếm số file ảnh hợp lệ
            image_count = len([f for f in os.listdir(subdir_path) 
                               if f.lower().endswith(VALID_EXTENSIONS)])
            
            print(f"{base:<15} | {subdir:<20} | {image_count:>12}")
            base_total += image_count
            
        print(f"{'-' * 55}")
        print(f"{'TỔNG CỘNG ' + base:<38} | {base_total:>12}")
        print(f"{'=' * 55}")
        total_all += base_total

    print(f"\n>>> TỔNG CỘNG TOÀN BỘ DATASET: {total_all} ảnh")

if __name__ == "__main__":
    # Danh sách các thư mục gốc của bạn
    MY_DATASETS = ["en", "vn"]
    count_images_in_dataset(MY_DATASETS)

Thư mục gốc     | Thư mục con          | Số lượng ảnh
-------------------------------------------------------
en              | 1_Anh_Nghieng        |          408
en              | 2_Anh_Cong           |            1
en              | 3_Anh_Mo             |          116
en              | 4_Chu_Nho_Day        |           16
en              | 5_Anh_Chuan          |           47
en              | 6_Khong_Co_Chu       |            0
-------------------------------------------------------
TỔNG CỘNG en                           |          588
vn              | 1_Anh_Nghieng        |            8
vn              | 2_Anh_Cong           |            5
vn              | 3_Anh_Mo             |            1
vn              | 4_Chu_Nho_Day        |          176
vn              | 5_Anh_Chuan          |           61
vn              | 6_Khong_Co_Chu       |            0
-------------------------------------------------------
TỔNG CỘNG vn                           |          251

>>> TỔNG CỘNG TOÀN BỘ

## Xử lý dữ liệu chuyên luận

In [1]:
import json

In [6]:
with open("database_duoc_thu_final.json", "r", encoding="utf-8") as f:
    data = json.load(f)
print(len(data))

667


In [5]:
all_keys = []
for i in data:
    all_keys.extend(i.keys())
    
print(f"Danh sách hoạt chất trong dữ liệu: {all_keys}" )

Danh sách hoạt chất trong dữ liệu: ['ABACAVIR', 'ACARBOSE', 'ACEBUTOLOL', 'ACENOCOUMAROL', 'ACETAZOLAMID', 'ACETYLCYSTEIN', 'ACICLOVIR', 'ACID ACETYLSALICYLIC', 'ACID AMINOCAPROIC', 'ACID ASCORBIC', 'ACID BORIC', 'ACID CHENODEOXYCHOLIC', 'ACID ETHACRYNIC', 'ACID FOLIC', 'ACID FUSIDIC', 'ACID IOPANOIC', 'ACID IOXAGLIC', 'ACID NALIDIXIC', 'ACID PANTOTHENIC', 'ACID PARA-AMINOBENZOIC', 'ACID SALICYLIC', 'ACID TRANEXAMIC', 'ACID IOPANOIC']


In [ ]:
import json

# 1. Load dữ liệu
with open("duoc_thu_final.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# 2. Tạo các dictionary để theo dõi: { tên_cần_kiểm_tra : tên_hoat_chat_gốc }
history_hoat_chat = {}
history_quoc_te = {}
history_thuong_mai = {}

def check_and_add(name_str, history_dict, current_drug_key, field_name):
    if not name_str or name_str.strip() == "":
        return
    
    # Xử lý trường hợp tên thương mại có nhiều tên cách nhau bởi dấu ";"
    names = [n.strip() for n in name_str.split(';')] if ';' in name_str else [name_str.strip()]
    
    for name in names:
        name_upper = name.upper() # Chuẩn hóa về chữ hoa để so sánh
        if name_upper in history_dict:
            print(f" TRÙNG LẶP [{field_name}]:")
            print(f"   - Giá trị: '{name}'")
            print(f"   - Xuất hiện tại: '{current_drug_key}'")
            print(f"   - Đã từng tồn tại ở: '{history_dict[name_upper]}'")
            print("-" * 30)
        else:
            history_dict[name_upper] = current_drug_key

# 3. Duyệt qua từng thuốc trong danh sách
for item in data:
    # Lấy key chính (ví dụ: "ACARBOSE")
    drug_key = list(item.keys())[0]
    details = item[drug_key]
    
    # Kiểm tra tên hoạt chất
    check_and_add(details.get("ten_hoat_chat"), history_hoat_chat, drug_key, "Tên hoạt chất")
    
    # Kiểm tra tên quốc tế
    check_and_add(details.get("ten_chung_quoc_te"), history_quoc_te, drug_key, "Tên quốc tế")
    
    # Kiểm tra tên thương mại (Xử lý chuỗi chứa nhiều tên cách bởi dấu ;)
    check_and_add(details.get("ten_thuong_mai"), history_thuong_mai, drug_key, "Tên thương mại")

print("Kiểm tra hoàn tất!")

⚠️ TRÙNG LẶP [Tên thương mại]:
   - Giá trị: 'Fosamax'
   - Xuất hiện tại: 'ALENDRONAT NATRI'
   - Đã từng tồn tại ở: 'ALENDRONAT NATRI'
------------------------------
⚠️ TRÙNG LẶP [Tên thương mại]:
   - Giá trị: 'Maiicaphami'
   - Xuất hiện tại: 'IOPANOIC'
   - Đã từng tồn tại ở: 'ACID BORIC'
------------------------------
Kiểm tra hoàn tất!


## xử lý file chuyên luận

In [ ]:
import json
import re

# 1. Cấu hình file và quy tắc
FILE_NAMES = [
    'database_duoc_thu_final.json', 
    'data.json', 

]

def clean_text(text, separator):
    """Làm sạch chuỗi, xóa dấu chấm cuối, gộp theo separator."""
    if not text or not isinstance(text, str):
        return ""
    # Tách theo các ký tự ngăn cách phổ biến
    parts = re.split(r'[;,\n|]', text)
    clean_parts = []
    for p in parts:
        p_clean = p.strip().strip('.')
        if p_clean and p_clean not in clean_parts:
            clean_parts.append(p_clean)
    return f"{separator} ".join(clean_parts)

def standardize_sub_keys(sub_dict):
    """
    Chuẩn hóa key 'Thời kỳ mang thai' -> 'thoi_ky_mang_thai'
    và 'Thời kỳ cho con bú' -> 'thoi_ky_cho_con_bu'
    """
    new_sub = {}
    if not isinstance(sub_dict, dict):
        return {}

    for k, v in sub_dict.items():
        # Chuyển key về chữ thường, bỏ dấu tiếng Việt cơ bản để nhận diện
        k_lower = k.lower()
        
        # Logic nhận diện Mang thai
        if "mang thai" in k_lower or "thai kỳ" in k_lower or "thoi_ky_mang_thai" in k_lower:
            target_key = "thoi_ky_mang_thai"
        # Logic nhận diện Cho con bú
        elif "cho con bú" in k_lower or "sua me" in k_lower or "thoi_ky_cho_con_bu" in k_lower:
            target_key = "thoi_ky_cho_con_bu"
        else:
            
            target_key = k_lower.replace(" ", "_")

        
        if isinstance(v, str):
            new_sub[target_key] = v.strip()
        else:
            new_sub[target_key] = v
            
    return new_sub

def merge_and_clean_data():
    merged_dict = {}

    for file_name in FILE_NAMES:
        try:
            with open(file_name, 'r', encoding='utf-8') as f:
                data = json.load(f)
                for item in data:
                    for drug_name, info in item.items():
                        # Dùng tên hoạt chất viết hoa làm Key duy nhất
                        main_key = drug_name.strip().upper()
                        
                        if main_key not in merged_dict:
                            merged_dict[main_key] = info
                        else:
                            # Gộp dữ liệu nếu trùng hoạt chất (tránh mất thông tin)
                            for field in ["ten_hoat_chat", "ten_chung_quoc_te", "ten_thuong_mai", "ma_atc"]:
                                old = merged_dict[main_key].get(field, "")
                                new = info.get(field, "")
                                if new and new not in old:
                                    merged_dict[main_key][field] = f"{old}, {new}"
                            
                            # Gộp cả object con
                            if "cac_truong_hop_cu_the" in info:
                                old_sub = merged_dict[main_key].get("cac_truong_hop_cu_the", {})
                                old_sub.update(info["cac_truong_hop_cu_the"])
                                merged_dict[main_key]["cac_truong_hop_cu_the"] = old_sub

        except Exception as e:
            print(f"Lỗi khi xử lý file {file_name}: {e}")

    # 2. Bước làm sạch cuối cùng (Final Polish)
    final_output = []
    for drug_key, info in merged_dict.items():
        clean_entry = {
            "ten_hoat_chat": clean_text(info.get("ten_hoat_chat", ""), ","),
            "ten_chung_quoc_te": clean_text(info.get("ten_chung_quoc_te", ""), ","),
            "ma_atc": clean_text(info.get("ma_atc", ""), ","),
            "ten_thuong_mai": clean_text(info.get("ten_thuong_mai", ""), ";"),
            "chong_chi_dinh": info.get("chong_chi_dinh", "").strip(),
            "than_trong": info.get("than_trong", "").strip(),
            "tuong_tac_thuoc": info.get("tuong_tac_thuoc", "").strip(),
            "cac_truong_hop_cu_the": standardize_sub_keys(info.get("cac_truong_hop_cu_the", {}))
        }
        final_output.append({drug_key: clean_entry})

    # 3. Xuất file
    with open('duoc_thu_master_clean.json', 'w', encoding='utf-8') as f:
        json.dump(final_output, f, ensure_ascii=False, indent=4)
    
    print(f"Hoàn thành! Đã gộp và chuẩn hóa {len(final_output)} hoạt chất.")

if __name__ == "__main__":
    merge_and_clean_data()

Hoàn thành! Đã gộp và chuẩn hóa 667 hoạt chất.


In [5]:
import json
import re
import unicodedata

FILE_NAMES = ['duoc_thu_master_clean.json']

def deep_clean_text(text, separator=","):
    """Làm sạch sâu: Chuẩn hóa Unicode, xóa khoảng trắng thừa, xử lý separator."""
    if not text or not isinstance(text, str):
        return ""
    
    # 1. Chuẩn hóa Unicode (NFC) để đồng nhất kiểu gõ tiếng Việt
    text = unicodedata.normalize('NFC', text)
    
    # 2. Xóa các ký tự xuống dòng và tab, thay bằng khoảng trắng
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    
    # 3. Tách chuỗi theo các dấu hiệu phân cách
    parts = re.split(r'[;,\n|]', text)
    
    clean_parts = []
    for p in parts:
        # Xóa dấu chấm cuối và khoảng trắng dư thừa
        p_clean = p.strip().strip('.')
        # Loại bỏ khoảng trắng thừa giữa các từ (ví dụ: "Abacavir   Sulfat" -> "Abacavir Sulfat")
        p_clean = " ".join(p_clean.split())
        
        if p_clean and p_clean not in clean_parts:
            clean_parts.append(p_clean)
            
    return f"{separator} ".join(clean_parts)

def standardize_sub_keys(sub_dict):
    """Chuẩn hóa các key trong cac_truong_hop_cu_the."""
    new_sub = {}
    if not isinstance(sub_dict, dict): return {}

    for k, v in sub_dict.items():
        k_clean = unicodedata.normalize('NFC', k).lower()
        
        # Nhận diện thông minh dựa trên từ khóa
        if any(word in k_clean for word in ["mang thai", "thai kỳ", "mang_thai"]):
            target_key = "thoi_ky_mang_thai"
        elif any(word in k_clean for word in ["cho con bú", "sua me", "cho_con_bu"]):
            target_key = "thoi_ky_cho_con_bu"
        else:
            target_key = k_clean.replace(" ", "_")

        # Làm sạch nội dung văn bản bên trong
        if isinstance(v, str):
            content = v.strip().replace('\n', ' ')
            new_sub[target_key] = " ".join(content.split())
        else:
            new_sub[target_key] = v
            
    return new_sub

def run_production_cleaning():
    merged_dict = {}

    for file_name in FILE_NAMES:
        try:
            with open(file_name, 'r', encoding='utf-8') as f:
                data = json.load(f)
                for item in data:
                    for drug_name, info in item.items():
                        # Chuẩn hóa tên hoạt chất chính (Key)
                        main_key = " ".join(drug_name.strip().upper().split())
                        
                        if main_key not in merged_dict:
                            merged_dict[main_key] = info
                        else:
                            # Gộp dữ liệu nếu trùng
                            for field in ["ten_hoat_chat", "ten_chung_quoc_te", "ten_thuong_mai", "ma_atc"]:
                                old = merged_dict[main_key].get(field, "")
                                new = info.get(field, "")
                                if new: merged_dict[main_key][field] = f"{old}, {new}"
                            
                            if "cac_truong_hop_cu_the" in info:
                                old_sub = merged_dict[main_key].get("cac_truong_hop_cu_the", {})
                                old_sub.update(info["cac_truong_hop_cu_the"])
                                merged_dict[main_key]["cac_truong_hop_cu_the"] = old_sub
        except Exception as e:
            print(f"Bỏ qua file {file_name} do lỗi: {e}")

    final_output = []
    for drug_key, info in merged_dict.items():
        # Xử lý làm sạch từng trường
        clean_entry = {
            "ten_hoat_chat": deep_clean_text(info.get("ten_hoat_chat", ""), ","),
            "ten_chung_quoc_te": deep_clean_text(info.get("ten_chung_quoc_te", ""), ","),
            "ma_atc": deep_clean_text(info.get("ma_atc", ""), ","),
            "ten_thuong_mai": deep_clean_text(info.get("ten_thuong_mai", ""), ";"),
            "chong_chi_dinh": " ".join(info.get("chong_chi_dinh", "").split()) or "Chưa có thông tin",
            "than_trong": " ".join(info.get("than_trong", "").split()) or "Chưa có thông tin",
            "tuong_tac_thuoc": " ".join(info.get("tuong_tac_thuoc", "").split()) or "Chưa có thông tin",
            "cac_truong_hop_cu_the": standardize_sub_keys(info.get("cac_truong_hop_cu_the", {}))
        }
        final_output.append({drug_key: clean_entry})

    with open('database_duoc_thu_final.json', 'w', encoding='utf-8') as f:
        json.dump(final_output, f, ensure_ascii=False, indent=4)
    
    print(f"Dữ liệu đã sẵn sàng! Tổng cộng: {len(final_output)} hoạt chất.")

run_production_cleaning()

Dữ liệu đã sẵn sàng! Tổng cộng: 667 hoạt chất.
